# Setup

In [ ]:
import ollama
from PIL import Image
import io
import base64
import os
import json
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from sentence_transformers import SentenceTransformer, util
from PIL import Image
import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from lancedb.rerankers import ColbertReranker
from tqdm.notebook import tqdm
import torch

#TODO: Create custom LanceDB query function which combines vector search for the image and text embeddings. Also check if we should use Hybrid with reranking or just vector search.

In [ ]:




# Configure pipeline to only extract embedded pictures
pipeline_options = PdfPipelineOptions()
pipeline_options.images_scale = 3.0   # adjust DPI scaling (2.0 ≈ 144 DPI)
pipeline_options.generate_picture_images = True
pipeline_options.generate_page_images = False  # <-- disable page renders

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# Load embedding models
image_embedding_model_name = "sentence-transformers/clip-ViT-B-32"
text_embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
registry = get_registry()
clip = registry.get("open-clip").create(device="cuda" if torch.cuda.is_available() else "cpu")
text_embedding_model = registry.get("huggingface").create(name=text_embedding_model_name, trust_remote_code=True, device="cuda" if torch.cuda.is_available() else "cpu")

study_names = [f for f in os.listdir("input") if f.endswith('.pdf')]
with open("image_chunks.json", "r", encoding="utf-8") as f:
    processed_chunks = json.load(f)

chunks_with_metadata = processed_chunks.copy()
processed_studies = set(chunk["document"] for chunk in processed_chunks)
study_names = [f for f in study_names if f not in processed_studies]
print(f"Found {len(processed_studies)} studies which are already processed.\nStudies which STILL need to be processed: {len(study_names)}:\n{study_names}...")

# Define DB Schema

In [ ]:
# Define model
class ImageModel(LanceModel):
    image_uri: str = clip.SourceField()  # As much as I dislike it, we should store the URI to the image instead of the image itself. This can also double as a unique ID.
    vector: Vector(clip.ndims()) = clip.VectorField()
    description: str = text_embedding_model.SourceField()
    description_vector: Vector(text_embedding_model.ndims()) = text_embedding_model.VectorField()
    document: str
    page: int

    @property
    def image(self):
        return Image.open(self.image_uri)

db = lancedb.connect("./db")
# db.create_table("my_image_table", schema=ImageModel, mode="overwrite") # Uncomment this line when running this cell for the first time
table = db.open_table("my_image_table")


# Process corpus for images

In [ ]:
for source in tqdm(study_names, desc="Chunking documents..."):
    doc = doc_converter.convert(f"input/{source}").document

    # Extract only embedded images (e.g., graphs/figures)
    for pic in tqdm(doc.pictures, desc=f"Processing images in {source[:20]}...", leave=False):
        if pic.image:  # each PictureItem has a PIL image
            pil_img = pic.image.pil_image
            buf = io.BytesIO() 
            pil_img.save(buf, format="PNG") # Convert image to raw bytes
            image_bytes = buf.getvalue()
            image_b64 = base64.b64encode(image_bytes).decode("utf-8")
            

            response = ollama.chat(
                model="gemma3:4b-it-qat",
                messages=[
                    {"role": "user", "content": "Is this image a logo/brand? If yes, output \"skip\". Else, describe the contents of the image. Try to limit your description to at most 200 word pieces.", "images": [image_b64]}
                ],
            )

            response = response["message"]['content'].strip()
            if "skip" == response.lower():
                print("Skipping logo/brand image.")
                continue
            else:

            # derive a filename (can include page number from provenance if needed)
                page = pic.prov[0].page_no if pic.prov else 0
                fname = f"{source}_page{page}_{pic.self_ref.split('/')[-1]}.png"
                pil_img.save(f"images/{fname}")

                image = {
                    "image_uri": f"images/{fname}",
                    "description": response,
                    "document": source,
                    "page": page,
                }
                chunks_with_metadata.append(image)


In [ ]:
with open("image_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks_with_metadata, f, ensure_ascii=False, indent=2)

In [ ]:
# Upload in batches with progress bar
batch_size = 100
for i in tqdm(range(0, len(chunks_with_metadata), batch_size), desc="Uploading chunks to VectorDB"):
    batch = chunks_with_metadata[i:i+batch_size]
    table.add(batch)

table.create_fts_index("description", replace=True) # Used by the reranker as well as the hybrid search's BM25 index
table.wait_for_index(["description_idx"])  # Wait for the indexing to finish

In [ ]:
db = lancedb.connect("./db")
table = db.open_table("my_image_table")
reranker = ColbertReranker(column="description")
# query_results

In [ ]:
for idx, row in df.iterrows():
    print(row["image_uri"])

# MULTIVECTOR SEARCH COMBINING CLIP IMAGE AND GEMMA DESCRIPTION EMBEDDINGS

In [ ]:
prompt = "A graph with a downward trend"
res1 = table.search(prompt, vector_column_name="description_vector").with_row_id(True).limit(5)
res2 = table.search(prompt, vector_column_name="vector").with_row_id(True).limit(5)
# print(type(res1), type(res2))
reranked = reranker.rerank_multivector([res1, res2], query=prompt, deduplicate=True)
reranked

for idx, row in reranked.to_pandas().iterrows():
    if idx < 3:
        display(Image.open(row['image_uri']))
        print(f"Description: {row['description']}\nDocument: {row['document']}, Page: {row['page']}\n")

# HYBRID SEARCH WITH CLIP IMAGE EMBEDDINGS AND FTS ON GEMMA DESCRIPTIONS

In [ ]:
query_results = table.search("A graph with a downward trend", query_type='hybrid',  vector_column_name="vector", fts_columns='description') \
                    .rerank(reranker) \
                    .limit(3).to_pydantic(ImageModel)

for result in query_results:
    display(result.image)
    print(f"Description: {result.description}\nDocument: {result.document}, Page: {result.page}\n")

# VECTOR SEARCH WITH GEMMA GENERATED DESCRIPTIONS

In [ ]:
query_results = table.search("A graph with a downward trend",  vector_column_name="description_vector") \
                    .rerank(reranker) \
                    .limit(5).to_pydantic(ImageModel)

for result in query_results:
    display(result.image)
    print(f"Description: {result.description}\nDocument: {result.document}, Page: {result.page}\n")

# HYBRID SEARCH WITH GEMMA GENERATED DESCRIPTION

In [ ]:
query_results = table.search("A graph with a downward trend", query_type='hybrid',  vector_column_name="description_vector", fts_columns="description") \
                    .rerank(reranker) \
                    .limit(3).to_pydantic(ImageModel)

for result in query_results:
    display(result.image)
    print(f"Description: {result.description}\nDocument: {result.document}, Page: {result.page}\n")

# VECTOR SEARCH WITH CLIP IMAGE EMBEDDINGS

In [ ]:
query_results = table.search("A graph with a downward trend",  vector_column_name="vector") \
                    .rerank(reranker) \
                    .limit(3).to_pydantic(ImageModel)

for result in query_results:
    display(result.image)
    print(f"Description: {result.description}\nDocument: {result.document}, Page: {result.page}\n")

# Docling picture extraction fun

In [ ]:
# from IPython.display import display
# display(image)
 
# # Convert bytes to a BytesIO buffer
# image_buffer = io.BytesIO(base64.b64decode(image_b64))

# # Open as a PIL image
# other_img = Image.open(image_buffer)
# display(other_img)


In [ ]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
# sample_text = """
# The image depicts a flowchart illustrating a process involving multiple data sources and analysis stages. It appears to represent a system for analyzing text data from various sources, including social media, web news, and potentially other data streams.

# The process begins with a collection of data from these sources. This raw data then passes through a Sentiment Analyzer, which presumably assesses the emotional tone or opinion expressed within the text. A Multi-source Instance Model (M-MI) seems to be central to this process, suggesting a model designed to handle and interpret information from multiple sources simultaneously.

# Furthermore, the flowchart includes Event Extraction, indicating a stage dedicated to identifying specific events mentioned within the text. The entire process culminates in an output, likely representing the processed and analyzed information. It's a visual representation of a complex data pipeline designed to derive insights from textual data.
# """
# num_tokens = len(tokenizer.encode(sample_text))
# print(f"Number of tokens: {num_tokens}")